In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
import torch
import json
import re
import numpy as np
import pandas as pd
import webcolors

In [ ]:
model_name = "/kaggle/input/subtask/transformers/default/6/codet5-large-lora-merged/codet5-large-lora/merged"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
_embed_model = None
def load_model():
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    return _embed_model
_embed_model = load_model()

In [ ]:
def flatten_json(obj, prefix=""):
    # Flatten JSON object thành dict {flattened_key: value}
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_prefix = f"{prefix}.{k}" if prefix else k
            out.update(flatten_json(v, new_prefix))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            new_prefix = f"{prefix}[{i}]"
            out.update(flatten_json(v, new_prefix))
    else:
        out[prefix] = obj
    return out

def greedy_max_matching(sim_matrix, keys1, keys2, threshold=0.7):
    sim_matrix = sim_matrix.copy()
    matched = []
    used_i, used_j = set(), set()
    
    while True:
        i, j = divmod(sim_matrix.argmax(), sim_matrix.shape[1])
        score = sim_matrix[i, j]
        if score < threshold:
            break
        if i in used_i or j in used_j:
            sim_matrix[i, j] = -1
            continue
        matched.append((keys1[i], keys2[j], score))
        used_i.add(i)
        used_j.add(j)
        sim_matrix[i, :] = -1
        sim_matrix[:, j] = -1
    # add phần chưa match
    for i in range(len(keys1)):
        if i not in used_i:
            matched.append((keys1[i], None, 0))
    for j in range(len(keys2)):
        if j not in used_j:
            matched.append((None, keys2[j], 0))
    return matched

In [ ]:
# ----- Parse numeric with unit -----
def parse_number(value: str):
    """Trả về (float_value, unit) hoặc (None, None)"""
    if isinstance(value, (int, float)):
        return float(value), ""
    if not isinstance(value, str):
        return None, None
    v = value.strip().lower()
    m = re.match(r"([-+]?\d*\.?\d+)([a-z%]*)", v)
    if m:
        num = float(m.group(1))
        unit = m.group(2) or ""
        return num, unit
    return None, None

In [ ]:
# ----- NUMBER (with unit tolerance) -----
def number_similarity(v1, v2):
    n1, u1 = parse_number(v1)
    n2, u2 = parse_number(v2)
    if n1 is None or n2 is None:
        return 0.0
    # Nếu đơn vị giống nhau hoặc 1 bên không có đơn vị -> vẫn coi như so sánh được
    if u1 == u2 or (u1 == "" or u2 == ""):
        return 1 - min(abs(n1-n2)/max(abs(n1),1e-5), 1)
    # Đơn vị khác nhau hẳn -> phạt mạnh
    return 0.0

In [ ]:
# ----- COLOR -----
def to_rgb(value: str):
    if not isinstance(value, str):
        return None
    value = value.strip().lower()
    if value.startswith("rgb"):
        nums = re.findall(r"\d+", value)
        if len(nums) == 3:
            return tuple(int(n) for n in nums)
    if value.startswith("#"):
        try:
            return webcolors.hex_to_rgb(value)
        except ValueError:
            return None
    try:
        return webcolors.name_to_rgb(value)
    except ValueError:
        return None
    return None

def color_similarity(c1, c2):
    rgb1, rgb2 = to_rgb(c1), to_rgb(c2)
    if rgb1 is None or rgb2 is None:
        return 0.0
    dist = np.linalg.norm(np.array(rgb1)-np.array(rgb2))
    max_dist = np.sqrt(255**2*3)
    return max(0, 1 - dist/max_dist)

In [ ]:
# ----- BOOLEAN -----
def bool_similarity(v1, v2):
    true_vals = {"true","yes","1","enabled","on"}
    false_vals = {"false","no","0","disabled","off"}
    v1, v2 = str(v1).lower(), str(v2).lower()
    if v1 in true_vals and v2 in true_vals: return 1.0
    if v1 in false_vals and v2 in false_vals: return 1.0
    if (v1 in true_vals and v2 in false_vals) or (v1 in false_vals and v2 in true_vals):
        return 0.0
    return 0.0

In [ ]:
# ----- TEXT -----
def text_similarity(v1, v2):
    model = load_model()
    emb = model.encode([str(v1), str(v2)], convert_to_tensor=True)
    return float(util.cos_sim(emb[0], emb[1]))

In [ ]:
# ----- AUTO-DETECT -----
def semantic_similarity(v1, v2):
    v1, v2 = str(v1).strip(), str(v2).strip()
    v1_low, v2_low = v1.lower(), v2.lower()

    # color
    if to_rgb(v1) and to_rgb(v2):
        return color_similarity(v1, v2)
    
    # number (px, %, plain int/float, mixed)
    n1, u1 = parse_number(v1)
    n2, u2 = parse_number(v2)
    if n1 is not None and n2 is not None:
        return number_similarity(v1, v2)

    # bool
    if v1_low in {"true","false","yes","no","enabled","disabled","on","off","0","1"} \
       and v2_low in {"true","false","yes","no","enabled","disabled","on","off","0","1"}:
        return bool_similarity(v1, v2)

    # fallback: text
    return text_similarity(v1, v2)

In [ ]:
def compare_values_with_keys(flat1, flat2, pairs, weightvalue=1):
    key_scores = []
    value_scores = []
    details = []

    for k1, k2, score_key in pairs:
        if k1 and k2:  # cả 2 key đều có
            v1, v2 = flat1.get(k1), flat2.get(k2)
            score_val = semantic_similarity(v1, v2)
            if score_val > 1:
                score_val = 1.0
            elif score_val < 0:
                score_val = 0.0
            value_scores.append(score_val)
            key_scores.append(score_key)
            details.append((f"{k1}:{v1}", f"{k2}:{v2}", score_key, score_val))

        elif k1 and not k2:  # key1 có, key2 không
            v1 = flat1.get(k1)
            key_scores.append(0.0)
            value_scores.append(0.0)
            details.append((f"{k1}:{v1}", "None", 0.0, 0.0))

        elif k2 and not k1:  # key2 có, key1 không -> bỏ qua
            v2 = flat2.get(k2)
            details.append(("None", f"{k2}:{v2}", None, None))
    details_sorted = sorted(details, key=lambda x: (x[2] if x[2] is not None else -1), reverse=True)

    keyscore = sum(key_scores) / len(key_scores)
    if keyscore < 0:
        keyscore = 0.0
    elif keyscore > 1:
        keyscore = 1.0

    valuescore = sum(value_scores) / len(value_scores)
    if valuescore < 0:
        valuescore = 0.0
    elif valuescore > 1:
        valuescore = 1.0

    avg = (keyscore + valuescore * weightvalue) / (1 + weightvalue)
    return details_sorted, keyscore, valuescore, avg

In [ ]:
# === Hàm parse JSON an toàn ===
def safe_json_parse(text: str):
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        # fallback: cố sửa ngoặc thừa thiếu
        fixed = text
        # đảm bảo số { == }
        while fixed.count("{") > fixed.count("}"):
            fixed += "}"
        while fixed.count("{") < fixed.count("}"):
            fixed = "{" + fixed
        try:
            return json.loads(fixed), True
        except:
            return text, False
            
        return text, False

In [ ]:
def generate_json_from_model(input_text, max_new_tokens=512):
    prompt = f"action2json: {input_text}"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
output = generate_json_from_model(
    "Verify student name list text contain 'a'", 
    max_new_tokens=512)
print(output)

In [ ]:
def process_subtask_file(file_path, threshold=0.7, weight_value=1, output_path="result.xlsx"):
    df = pd.read_excel(file_path)

    results = []
    for i, row in df.iterrows():
        print(i)
        
        subtask = row["Sub Task"]
        step_json_str = row["Step Object"]

        # Parse JSON mẫu
        try:
            step_json = json.loads(step_json_str)
        except:
            step_json = {}

        # Sinh JSON từ model
        try:
            output = generate_json_from_model(subtask, max_new_tokens=512)
        except:
            output = "{}"

        # Kiểm tra valid JSON
        parsed, is_valid = safe_json_parse(output)

        if not is_valid:
            results.append({
                "Sub Task": subtask,
                "Step Object": step_json_str,
                "Model": output,
                "Valid_JSON": False,
                "Result": "Fail",
                "AVG Score": 0.0,
                "Details": "",
                "Key Scores": 0.0,
                "Value Scores": 0.0
            })
            continue

        # So sánh JSON mẫu vs JSON sinh
        flat1 = flatten_json(step_json)
        flat2 = flatten_json(parsed)

        keys1, keys2 = list(flat1.keys()), list(flat2.keys())
        emb1 = load_model().encode(keys1, convert_to_tensor=True)
        emb2 = load_model().encode(keys2, convert_to_tensor=True)
        sim_matrix = util.cos_sim(emb1, emb2).cpu().numpy()

        pairs = greedy_max_matching(sim_matrix, keys1, keys2, threshold=threshold)
        details, key_score, value_score, avg = compare_values_with_keys(flat1, flat2, pairs)
        
        if avg > 1.0:
            avg = 1.0
        elif avg < 0.0:
            avg = 0.0
        result = "Pass" if avg >= threshold else "Fail"
        detail_str = "\n".join([
            f"{l} <-> {r} (key_score={ks:.3f}) (value_score={vs:.3f})" if ks is not None else f"{l} <-> {r} (skip)"
            for l, r, ks, vs in details
        ])
        
        results.append({
            "Sub Task": subtask,
            "Step Object": step_json_str,
            "Model": output,
            "Valid_JSON": True,
            "Result": result,
            "AVG Score": avg,
            "Details": detail_str,
            "Key Scores": key_score,
            "Value Scores": value_score
        })

    out_df = pd.DataFrame(results)
    out_df.to_excel(output_path, index=False)
    print(f"[Done] Saved result to {output_path}")
    return out_df

In [ ]:
result = process_subtask_file(
    "/kaggle/input/data-chuan/subtask_and_step.xlsx",
    threshold=0.7,
    weight_value=1,
    output_path="result.xlsx"
)
result.head()